# Dupin · Ingesta de PaySim → GCS

**Fase 0/1 · plano offline (Colab).** Descarga PaySim desde Kaggle directamente a
la VM de Colab y lo sube a GCS. *Sin pasar por la máquina local del usuario*: el
dato viaja máquina-a-máquina (Kaggle → Colab → GCS) y nunca toca tu equipo.

El bucket `raw/` guarda el CSV **crudo e inmutable**. No se reescribe.

---

**Dataset:** PaySim — Lopez-Rojas, Elmir & Axelsson (2016), EMSS. **CC BY-SA 4.0.**
Kaggle: `ealaxi/paysim1`. El CSV nunca entra al repo git (Invariante 8).

## 1. Dependencias

In [ ]:
!pip -q install kaggle google-cloud-storage pandas

## 2. Configuración

Ajusta a tu proyecto. Coincide con `config.example.yaml`.

In [ ]:
PROJECT_ID   = "dupin-dupin"
BUCKET_RAW   = "dupin-dupin-raw"
KAGGLE_DATASET = "ealaxi/paysim1"

# Nombre del CSV dentro del zip de Kaggle y destino en GCS.
CSV_NAME   = "PS_20174392719_1491204439457_log.csv"
RAW_OBJECT = "raw/paysim/" + CSV_NAME          # gs://dupin-dupin-raw/raw/paysim/...
LOCAL_DIR  = "/content/paysim"

## 3. Autenticación de GCP

Usa la identidad de tu sesión de Colab (la misma cuenta Google del proyecto).

In [ ]:
from google.colab import auth
auth.authenticate_user()
!gcloud config set project {PROJECT_ID}
print("Proyecto activo:", PROJECT_ID)

## 4. Credenciales de Kaggle

Guarda tu API token de Kaggle en **Colab Secrets** (icono de llave 🔑) con nombres
`KAGGLE_USERNAME` y `KAGGLE_KEY`. Así no se escriben en el notebook ni se versionan.
Obtén el token en kaggle.com → Account → *Create New API Token*.

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
print("Credenciales de Kaggle cargadas para:", os.environ["KAGGLE_USERNAME"])

## 5. Descarga PaySim (Kaggle → VM de Colab)

In [ ]:
import os
os.makedirs(LOCAL_DIR, exist_ok=True)
!kaggle datasets download -d {KAGGLE_DATASET} -p {LOCAL_DIR} --unzip
print("Contenido descargado:")
!ls -lh {LOCAL_DIR}

## 6. Subida a GCS (`raw/`, inmutable)

Sube el CSV crudo al bucket `raw`. Si el objeto ya existe, **no se sobrescribe**:
`raw/` es inmutable por contrato. Para re-ingestar, borra el objeto a mano y
documenta por qué.

In [ ]:
from google.cloud import storage

local_csv = os.path.join(LOCAL_DIR, CSV_NAME)
assert os.path.exists(local_csv), f"No se encontró {local_csv}; revisa CSV_NAME."

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_RAW)
blob = bucket.blob(RAW_OBJECT)

if blob.exists():
    print("Ya existe (inmutable, no se sobrescribe):", f"gs://{BUCKET_RAW}/{RAW_OBJECT}")
else:
    print("Subiendo... (CSV ~470 MB, puede tardar)")
    blob.upload_from_filename(local_csv)
    print("Subido:", f"gs://{BUCKET_RAW}/{RAW_OBJECT}")

## 7. Verificación

In [ ]:
import pandas as pd

blob.reload()
print(f"gs://{BUCKET_RAW}/{RAW_OBJECT}")
print(f"Tamaño: {blob.size/1e6:,.1f} MB")

# Lectura de cabecera para confirmar esquema (solo primeras filas).
head = pd.read_csv(f"gs://{BUCKET_RAW}/{RAW_OBJECT}", nrows=5)
print("\nColumnas:", list(head.columns))
head

---
**Listo.** PaySim vive ahora en `gs://dupin-dupin-raw/raw/paysim/` como dato crudo
inmutable. Continúa con `notebooks/01_fraud_exploration.ipynb` (Fase 1).